# 1 Importing and Merging Dataset

In [2]:
import pandas as pd
from fitparse import FitFile
from collections import Counter
from pathlib import Path
from datetime import datetime, timedelta

In [3]:
from fitparse import FitFile

fitfile = FitFile("../data/444053264780_WELLNESS.fit")

for i, msg in enumerate(fitfile.get_messages("stress_level")):
    print(msg.get_values())

    if i == 5:
        break

{'stress_level_time': datetime.datetime(2026, 5, 31, 8, 36), 'stress_level_value': 22, 'unknown_2': 11, 'unknown_3': 74, 'unknown_4': 1}
{'stress_level_time': datetime.datetime(2026, 5, 31, 8, 37), 'stress_level_value': 36, 'unknown_2': -15, 'unknown_3': 74, 'unknown_4': 1}
{'stress_level_time': datetime.datetime(2026, 5, 31, 8, 38), 'stress_level_value': 34, 'unknown_2': -12, 'unknown_3': 74, 'unknown_4': 1}
{'stress_level_time': datetime.datetime(2026, 5, 31, 8, 39), 'stress_level_value': 42, 'unknown_2': -22, 'unknown_3': 74, 'unknown_4': 1}
{'stress_level_time': datetime.datetime(2026, 5, 31, 8, 40), 'stress_level_value': 33, 'unknown_2': -10, 'unknown_3': 74, 'unknown_4': 1}
{'stress_level_time': datetime.datetime(2026, 5, 31, 8, 41), 'stress_level_value': 36, 'unknown_2': -15, 'unknown_3': 74, 'unknown_4': 1}


### Stress

In [5]:
rows = []

for msg in fitfile.get_messages("stress_level"):
    row = msg.get_values()
    rows.append({
        "timestamp": row["stress_level_time"],
        "stress": row["stress_level_value"]
    })

stress_df = pd.DataFrame(rows)
stress_df["stress"] = stress_df["stress"].replace([-1, -2], pd.NA)

stress_df.head()

,timestamp,stress
0,2026-05-31 08:36:00,22
1,2026-05-31 08:37:00,36
2,2026-05-31 08:38:00,34
3,2026-05-31 08:39:00,42
4,2026-05-31 08:40:00,33


In [6]:
stress_df.tail()

,timestamp,stress
610,2026-05-31 19:04:00,36
611,2026-05-31 19:05:00,33
612,2026-05-31 19:06:00,30
613,2026-05-31 19:07:00,42
614,2026-05-31 19:08:00,49


### Heart Rate

In [8]:
rows = []

current_date = None

for msg in fitfile.get_messages("monitoring"):
    row = msg.get_values()

    # Update current date whenever Garmin provides a full timestamp
    if "timestamp" in row:
        current_date = row["timestamp"].date()

    # Heart-rate records
    if "heart_rate" in row and "timestamp_16" in row and current_date is not None:

        seconds = row["timestamp_16"]

        timestamp = datetime.combine(
            current_date,
            datetime.min.time()
        ) + timedelta(seconds=seconds)

        rows.append({
            "timestamp": timestamp,
            "heart_rate": row["heart_rate"]
        })

heart_rate_df = pd.DataFrame(rows)

heart_rate_df.head()

,timestamp,heart_rate
0,2026-05-31 11:51:16,67
1,2026-05-31 11:52:16,69
2,2026-05-31 11:53:16,70
3,2026-05-31 11:54:16,73
4,2026-05-31 11:55:16,74


In [9]:
heart_rate_df.tail()

,timestamp,heart_rate
556,2026-05-31 04:08:00,77
557,2026-05-31 04:09:00,73
558,2026-05-31 04:10:00,72
559,2026-05-31 04:11:00,73
560,2026-05-31 04:12:00,75


### Body Battery

In [11]:
rows = []

for msg in fitfile.get_messages("stress_level"):
    row = msg.get_values()

    rows.append({
        "timestamp": row["stress_level_time"],
        "body_battery": row["unknown_3"]
    })

body_battery_df = pd.DataFrame(rows)
body_battery_df["body_battery"] = body_battery_df["body_battery"].replace([-1, -2], pd.NA)

body_battery_df.head()

,timestamp,body_battery
0,2026-05-31 08:36:00,74
1,2026-05-31 08:37:00,74
2,2026-05-31 08:38:00,74
3,2026-05-31 08:39:00,74
4,2026-05-31 08:40:00,74


In [12]:
body_battery_df.tail()

,timestamp,body_battery
610,2026-05-31 19:04:00,22
611,2026-05-31 19:05:00,22
612,2026-05-31 19:06:00,22
613,2026-05-31 19:07:00,22
614,2026-05-31 19:08:00,21


### Respirational Rate

In [14]:
GARMIN_EPOCH = datetime(1989, 12, 31)

rows = []

for msg in fitfile.get_messages("unknown_297"):
    row = msg.get_values()

    rows.append({
        "timestamp": GARMIN_EPOCH + timedelta(seconds=row["unknown_253"]),
        "respiration_rate": row["unknown_0"] / 100
    })

respiration_df = pd.DataFrame(rows)

# Garmin missing values
respiration_df.loc[
    respiration_df["respiration_rate"] < 0,
    "respiration_rate"
] = pd.NA

respiration_df.head()

,timestamp,respiration_rate
0,2026-05-31 08:35:00,17.66
1,2026-05-31 08:36:00,18.25
2,2026-05-31 08:37:00,17.58
3,2026-05-31 08:38:00,19.16
4,2026-05-31 08:39:00,18.08


In [15]:
respiration_df.tail()

,timestamp,respiration_rate
629,2026-05-31 19:04:00,17.00
630,2026-05-31 19:05:00,16.41
631,2026-05-31 19:06:00,15.08
632,2026-05-31 19:07:00,18.25
633,2026-05-31 19:08:00,19.16


### Merging

In [17]:
# Make sure timestamps are datetime
for df in [stress_df, body_battery_df, respiration_df, heart_rate_df]:
    df["timestamp"] = pd.to_datetime(df["timestamp"])

# Pick the day from the data
day = stress_df["timestamp"].dt.normalize().iloc[0]

# Full 1-minute timeline: 00:00 to 23:59
full_index = pd.date_range(
    start=day,
    end=day + pd.Timedelta(days=1) - pd.Timedelta(minutes=1),
    freq="1min"
)

def prepare_minute_df(df, value_col):
    temp = df.copy()
    
    # Round/floor timestamps down to minute
    temp["timestamp"] = temp["timestamp"].dt.floor("min")
    
    # If duplicate values exist in the same minute, average them
    temp = (
        temp.groupby("timestamp", as_index=True)[value_col]
        .mean()
        .to_frame()
    )
    
    return temp

stress_min = prepare_minute_df(stress_df, "stress")
body_battery_min = prepare_minute_df(body_battery_df, "body_battery")
respiration_min = prepare_minute_df(respiration_df, "respiration_rate")
heart_rate_min = prepare_minute_df(heart_rate_df, "heart_rate")

# Combine into one standardized dataframe
daily_df = pd.DataFrame(index=full_index)

daily_df = daily_df.join(stress_min)
daily_df = daily_df.join(body_battery_min)
daily_df = daily_df.join(respiration_min)
daily_df = daily_df.join(heart_rate_min)

daily_df.index.name = "timestamp"

# Optional: reset index if you want timestamp as a normal column
daily_df = daily_df.reset_index()

daily_df

,timestamp,stress,body_battery,respiration_rate,heart_rate
0,2026-05-31 00:00:00,NaN,NaN,NaN,NaN
1,2026-05-31 00:01:00,NaN,NaN,NaN,84.0
2,2026-05-31 00:02:00,NaN,NaN,NaN,90.0
3,2026-05-31 00:03:00,NaN,NaN,NaN,95.0
4,2026-05-31 00:04:00,NaN,NaN,NaN,92.0
...,...,...,...,...,...
1435,2026-05-31 23:55:00,NaN,NaN,NaN,NaN
1436,2026-05-31 23:56:00,NaN,NaN,NaN,NaN
1437,2026-05-31 23:57:00,NaN,NaN,NaN,NaN
1438,2026-05-31 23:58:00,NaN,NaN,NaN,NaN


# 2 EDA

### Missing Values

In [20]:
daily_df.isna().sum()

timestamp             0
stress              968
body_battery        825
respiration_rate    952
heart_rate          879
dtype: int64